In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report


In [2]:
# 🧱 Step 1 — Load & Prepare Data
# Load your enriched dataset
df = pd.read_parquet("D:\LPA_MTech_Project\Enriched_Datasets\SupremeCourt_Combined_2020_2025_enriched.parquet")

# Keep only rows with known verdicts
df = df.dropna(subset=['verdict_label'])

# Basic cleaning
df['clean_text'] = df['text'].astype(str).str.replace(r'\s+', ' ', regex=True).str.lower()

# Select useful features
text_col = 'clean_text'
num_cols = ['bench_size', 'sentiment_score', 'num_citations', 'pet_vs_resp_ratio', 'num_unique_acts']
target_col = 'verdict_label'


In [3]:
# 🧠 Step 2 — Feature Engineering (TF-IDF + Numeric)
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF setup
tfidf = TfidfVectorizer(max_features=8000, ngram_range=(1,2), stop_words='english')
X_text = tfidf.fit_transform(df[text_col])

# Numeric features
X_num = df[num_cols].fillna(0).values
scaler = StandardScaler()
X_num = scaler.fit_transform(X_num)

# Combine sparse + dense features
X_combined = np.hstack((X_text.toarray(), X_num))

y = df[target_col].astype(int).values

# Split
X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, stratify=y, random_state=42)


In [4]:
# ⚙️ Step 3 — Prepare PyTorch Tensors (GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Convert to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)




Using device: cuda


In [5]:
# 🧩 Step 4 — Define Simple Neural Network (GPU-compatible)


class VerdictNet(nn.Module):
    def __init__(self, input_dim):
        super(VerdictNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.net(x)

model = VerdictNet(input_dim=X_train.shape[1]).to(device)


In [6]:
# ⚡ Step 5 — Training Loop
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")


Epoch 1/10 - Loss: 0.5470
Epoch 2/10 - Loss: 0.4366
Epoch 3/10 - Loss: 0.2823
Epoch 4/10 - Loss: 0.1434
Epoch 5/10 - Loss: 0.0667
Epoch 6/10 - Loss: 0.0381
Epoch 7/10 - Loss: 0.0104
Epoch 8/10 - Loss: 0.0048
Epoch 9/10 - Loss: 0.0022
Epoch 10/10 - Loss: 0.0022


In [7]:
# 🧪 Step 6 — Evaluation

model.eval()
y_pred_list = []

with torch.no_grad():
    for X_batch, _ in test_loader:
        X_batch = X_batch.to(device)
        preds = model(X_batch)
        y_pred = torch.argmax(preds, dim=1)
        y_pred_list.extend(y_pred.cpu().numpy())

acc = accuracy_score(y_test, y_pred_list)
print("\nTest Accuracy:", acc)
print("\nClassification Report:\n", classification_report(y_test, y_pred_list))
    


Test Accuracy: 0.7712177121771218

Classification Report:
               precision    recall  f1-score   support

           0       0.54      0.45      0.49       201
           1       0.83      0.88      0.85       612

    accuracy                           0.77       813
   macro avg       0.69      0.66      0.67       813
weighted avg       0.76      0.77      0.76       813



In [8]:
# 💾 Step 7 — Save Model & Vectorizer
torch.save(model.state_dict(), "My_Models/verdict_predictor.pt")

import joblib
joblib.dump(tfidf, "My_Models/tfidf_vectorizer.pkl")
joblib.dump(scaler, "My_Models/num_scaler.pkl")


['My_Models/num_scaler.pkl']

In [9]:
# 🧠 Optional: Compare With Classical Models (CPU)
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
print("LogReg Accuracy:", clf.score(X_test, y_test))



LogReg Accuracy: 0.7589175891758918
